# E17b · Level-2A training images from midsummer

E17a trained on Level-2A images of the EuroSAT patches and reached 71.97% alone, but as the fourth ensemble member it
gave the new best result (E15d, **72.95%**). Its downloads were clearly less green than Kaggle (median image NDVI about
0.52 vs 0.69; brighter B04, B11, B12): most scenes were from September, often after harvest, and 2018 was a drought
summer in central Europe.

E17b downloads a second set with **June to August only, from 2017, 2019 and 2020** (no 2018):

```
python scripts/download_l2a.py --tag summer --months 6-8 \
    --dates 2017-06-01/2017-08-31,2019-06-01/2019-08-31,2020-06-01/2020-08-31
```

If the gap to Kaggle was mostly season, the band medians in section 2 should move towards Kaggle and E17b should beat
E17a, alone and in the ensemble. Everything else is exactly E17a (E09a setup, round-3 pseudo-labels, no haze
correction). One change: the scene dates.


## 1 · Setup

In [ ]:
import json
import sys
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))
from src.data import CLASSES, PROJECT_ROOT, TEST_BANDS, load_split, load_test, load_train, load_train_l2a, train_metadata
from src.experiment import (add_pseudo_labels, class_mix, compare_variants, find_run, finish_run, kaggle_monitor,
                            kaggle_score, load_correction, load_experiment_data, load_saved_model, plot_class_mix,
                            plot_variant_curves, select_view_agree, teacher_predictions, view_agreement)
from src.models import MEDIUM_TRACK_LIMIT, build_model
from src.plot_style import apply_style
from src.training import Normalizer, TrainConfig, fit, load_run, predict_views
from src.utils import describe_device, get_device, seed_everything

apply_style()

# ---- Experiment settings ---------------------------------------------------------------
NOTEBOOK = "E17b_l2a_summer"
LOAD_RUNS = {   # saved runs; None = train this variant again
    "E17b": None,
}
REFERENCE = "pl_r3_view_agree_contrast_aug_hist_split_effb0"               # E09a, no AdaBN, 72.67%
E17A = "l2a_pl_r3_view_agree_contrast_aug_effb0"                             # E17a, no AdaBN, 71.97%
SOURCE = "l2a_summer"
R3_TEACHER = "adabn_balanced_pseudo_label_r2_x2_contrast_aug_hist_split_effb0"   # E06c + AdaBN: round-3 labels
CAPS = {"SeaLake": 400}
REPEAT = 2
MODEL = "efficientnet_b0"
BANDS = list(TEST_BANDS)
SEED = 42
cfg = TrainConfig(epochs=20, batch_size=128, lr=1e-3, weight_decay=1e-4,
                  label_smoothing=0.1, patience=5, augment=True,
                  photometric={"contrast": (0.9, 1.5), "band_contrast": (0.9, 1.15),
                               "brightness": (0.93, 1.07), "p": 0.8})   # as in E09a
# -----------------------------------------------------------------------------------------

seed_everything(SEED)
device = get_device()
print("Device:", describe_device(device))
results = {}


def base_config(variant, run_name, description, extra):
    return {"run_name": run_name, "description": description, "notebook": NOTEBOOK, "variant": variant,
            "model": MODEL, "bands": BANDS, "corrections": [], "source": SOURCE, "train": asdict(cfg),
            "split": "split_seed42", **extra}

## 2 · How close are the Level-2A downloads to the Kaggle images?

Per-band median over images and the median image NDVI: EuroSAT after `hist_split` (E09a), the September-heavy set
(E17a), the midsummer set (E17b) and Kaggle. E17b should be closer to Kaggle than E17a, above all in B04, B11, B12
and NDVI. Then a few patches side by side (RGB): the label should still fit the Level-2A image.

In [ ]:
X_l1c, _ = load_train()
X_a, ok_a = load_train_l2a()                 # E17a set
X_b, ok = load_train_l2a(tag="summer")       # E17b set
X_kg, _ = load_test()
meta = train_metadata()
print(f"Midsummer Level-2A available: {ok.sum():,} of {len(ok):,} patches ({ok.mean():.1%})")
display(pd.Series(ok, index=meta["label"]).groupby(level=0).mean().reindex(CLASSES).mul(100).round(1)
        .rename("% with Level-2A").to_frame().T)

rng = np.random.default_rng(SEED)
sample = np.sort(rng.choice(np.flatnonzero(ok & ok_a), 3000, replace=False))
sets = {"L1C + hist_split (E09a)": load_correction("hist_split")(np.asarray(X_l1c[sample]).copy()),
        "L2A Apr-Oct (E17a)": np.asarray(X_a[sample]), "L2A Jun-Aug (E17b)": np.asarray(X_b[sample]),
        "Kaggle L2A": np.asarray(X_kg[rng.choice(len(X_kg), 3000, replace=False)])}
img_means = {k: v.reshape(len(v), 12, -1).astype(np.float32).mean(2) for k, v in sets.items()}
b4, b8 = TEST_BANDS.index("B04"), TEST_BANDS.index("B08")
bands = pd.DataFrame({k: np.median(m, axis=0) for k, m in img_means.items()}, index=TEST_BANDS).round(0)
bands.loc["NDVI"] = [np.median((m[:, b8] - m[:, b4]) / (m[:, b8] + m[:, b4] + 1)) for m in img_means.values()]
bands.round(2)


In [ ]:
def rgb(x):
    img = x[[TEST_BANDS.index(b) for b in ["B04", "B03", "B02"]]].transpose(1, 2, 0).astype(np.float32)
    return np.clip(img / 2500, 0, 1) ** 0.8

show = [rng.choice(np.flatnonzero(ok & ok_a & (meta["label"].values == c))) for c in CLASSES]
fig, axes = plt.subplots(3, len(CLASSES), figsize=(2 * len(CLASSES), 6.4))
for j, i in enumerate(show):
    axes[0, j].imshow(rgb(np.asarray(X_l1c[i]))); axes[0, j].set_title(CLASSES[j], fontsize=8)
    axes[1, j].imshow(rgb(np.asarray(X_a[i])))
    axes[2, j].imshow(rgb(np.asarray(X_b[i])))
for ax in axes.ravel():
    ax.axis("off")
axes[0, 0].text(-8, 32, "L1C", rotation=90, va="center"); axes[1, 0].text(-8, 32, "E17a", rotation=90, va="center"); axes[2, 0].text(-8, 32, "E17b", rotation=90, va="center")
fig.suptitle("Same patch: EuroSAT L1C, L2A April-October (E17a), L2A June-August (E17b)")
plt.tight_layout()
del X_l1c, X_a, sets, img_means

## 3 · Data and round-3 pseudo-labels (as in E09a)

In [ ]:
data = load_experiment_data(BANDS, [], device, source=SOURCE)
print(data.summary())
table, _ = teacher_predictions(R3_TEACHER, data, device)   # Kaggle images only, same labels as E09a
hard_idx = select_view_agree(table, CAPS, SEED)
print(f"{len(hard_idx):,} pseudo-labels (E09a: 2,948)")

## 4 · E17b · E17a with midsummer Level-2A images

In [ ]:
tr_images, tr_labels, _ = add_pseudo_labels(data, table, hard_idx, repeat=REPEAT)
if LOAD_RUNS["E17b"]:
    model, normalizer, history = load_saved_model(LOAD_RUNS["E17b"], device)
    best_state = None
else:
    seed_everything(SEED)
    normalizer = Normalizer.fit(data.X_tr)
    model = build_model(MODEL, bands=BANDS, pretrained=True).to(device)
    history, best_state = fit(model, tr_images, tr_labels, data.va_images, data.va_labels,
                              normalizer, cfg, monitor=kaggle_monitor(data, normalizer))

results["E17b"] = finish_run(model, normalizer, history, best_state, data, variant="E17b", load_run=LOAD_RUNS["E17b"],
    device=device, seed=SEED, param_limit=MEDIUM_TRACK_LIMIT,
    config=base_config("E17b", "l2a_summer_pl_r3_view_agree_contrast_aug_effb0",
                       "E17a with midsummer Level-2A images (June-August 2017, 2019, 2020)",
                       {"n_l2a_patches": int(data.X_tr.shape[0] + data.X_va.shape[0] + data.X_te.shape[0]),
                        "pseudo_labels": {"teacher": R3_TEACHER, "selection": "view_agree", "caps": CAPS,
                                          "repeat": REPEAT, "n_images": int(len(hard_idx))}}))
del tr_images, tr_labels

## 5 · Comparison with E09a and E17a

E17b's first row is measured with AdaBN; the other rows without. E17a was best without AdaBN (71.97% vs 70.61%).

In [ ]:
ref_dir = find_run(REFERENCE)
r_model, r_norm, _ = load_run(ref_dir, device)
ref_views = predict_views(r_model, data.kg_images, r_norm)
ref_prob = ref_views.mean(0)
del r_model
summary = compare_variants(results, reference=ref_prob, reference_name="E09a")
r_metrics = json.loads((ref_dir / "metrics.json").read_text())
a_metrics = json.loads((find_run("adabn_balanced_" + REFERENCE) / "metrics.json").read_text())
summary.loc["E09a (no AdaBN)"] = pd.Series({"description": "best single model", "confidence (%)": ref_prob.max(1).mean() * 100,
                                            "view agreement (%)": view_agreement(ref_views) * 100,
                                            "same class as E09a (%)": 100.0, "Kaggle public (%)": kaggle_score(ref_dir)})
summary.loc["E09a (AdaBN)"] = pd.Series({"description": "best single model + AdaBN",
                                         "confidence (%)": a_metrics["kaggle_mean_confidence"] * 100,
                                         "view agreement (%)": a_metrics["kaggle_view_agreement"] * 100,
                                         "Kaggle public (%)": kaggle_score(find_run("adabn_balanced_" + REFERENCE))})
for k, r in results.items():
    summary.loc[k + " (no AdaBN)"] = pd.Series({"description": "no AdaBN", "confidence (%)": r.kg_prob.max(1).mean() * 100,
                                                "same class as E09a (%)": (r.kg_prob.argmax(1) == ref_prob.argmax(1)).mean() * 100,
                                                "Kaggle public (%)": kaggle_score(r.run_dir)})
summary.round(2)
a_dir = find_run(E17A)
a_model, a_norm, _ = load_run(a_dir, device)
a_views = predict_views(a_model, data.kg_images, a_norm)
del a_model
summary.loc["E17a (no AdaBN)"] = pd.Series({"description": "L2A April-October", "confidence (%)": a_views.mean(0).max(1).mean() * 100,
                                            "view agreement (%)": view_agreement(a_views) * 100,
                                            "same class as E09a (%)": (a_views.mean(0).argmax(1) == ref_prob.argmax(1)).mean() * 100,
                                            "Kaggle public (%)": kaggle_score(a_dir)})
summary.round(2)

In [ ]:
# Different predictions: E09a's class (rows) -> E17b's class, no AdaBN (columns)
ref_pred = np.array(CLASSES)[ref_prob.argmax(1)]
for k, r in results.items():
    new = np.array(CLASSES)[r.kg_prob.argmax(1)]
    changed = new != ref_pred
    print(f"{k}: {changed.sum()} images differ from E09a ({changed.mean():.1%})")
    display(pd.crosstab(ref_pred[changed], new[changed]).rename_axis(index="E09a", columns=k))

In [ ]:
plot_variant_curves(results)
plot_class_mix(pd.DataFrame({"train split": class_mix(np.eye(10)[data.y_tr]), "E09a": class_mix(ref_prob),
                             **{k: class_mix(r.prob_ada) for k, r in results.items()}}))

## 6 · Conclusion

*To fill in after the Kaggle submission (both files: with and without AdaBN).*

| Model | Training images | Kaggle public (no AdaBN / AdaBN) |
|---|---|---|
| E09a | EuroSAT L1C + hist_split | 72.67% / 71.78% |
| E17a | L2A April-October 2017-2018 | 71.97% / 70.61% |
| E17b | L2A June-August 2017, 2019, 2020 | |
| E15d | ensemble E09a, E12a, E13a, E17a | **72.95%** |

If E17b beats E17a, put it in the ensemble in place of (or next to) E17a (E15 notebook).
